# 07. Математика π-Flow: от affine path к DX policy

<!-- study-navigation:start -->
**Основа:** [MIT 6.S184 / 2026](https://diffusion.csail.mit.edu/2026/index.html) · [Основной конспект](https://diffusion.csail.mit.edu/2026/docs/lecture_notes.pdf) · [Учебная карта](../../curriculum/README.md) · [Полное содержание](../../curriculum/CONTENTS.md).

**Связанные лекции:** [Lecture 1](https://diffusion.csail.mit.edu/2026/docs/20260120_Lecture_01.pdf) · [Lecture 2](https://diffusion.csail.mit.edu/2026/docs/20260122_Lecture_02.pdf) · [Lecture 3](https://diffusion.csail.mit.edu/2026/docs/20260123_Lecture_03.pdf).
<!-- study-navigation:end -->

Выведем u→x0, raw/shifted time, grid interpolation, polynomial mode, gradients/detach и discrete policy rollout для каждого метода piflow_math.py.

[Маршрут](../README.md) · [Точные задания MIT](../MIT_BRIDGE.md) · [Методы и source map](../SOURCE_MAP.md) · [Измерения](../reports/VALIDATION.md).

Каждый опыт: **теория → формула → формы/единицы → прогноз → код → измерение → ограниченный вывод**. CPU по умолчанию; без pretrained weights, сети и ComfyUI jobs.


**Связь с MIT:** MIT Lab 2 Q2.1–Q2.3 и notes §3.1–3.2 дают affine path и conditional velocity. DX reparameterization выводится из них; π-Flow policy, time warp и interpolation — отдельное исследовательское расширение.

[MIT course](https://diffusion.csail.mit.edu/2026/index.html) · [Основной конспект](https://diffusion.csail.mit.edu/2026/docs/lecture_notes.pdf).


In [ ]:
from pathlib import Path
import sys,math,json,copy
import numpy as np
import torch
from torch import nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
COURSE=next((p/'dit' for p in [Path.cwd(),*Path.cwd().parents] if (p/'dit/labkit/core.py').is_file()),None)
assert COURSE is not None,'Clone the study repository first; see dit/README.md (Colab setup).'
sys.path.insert(0,str(COURSE))
from labkit.core import (native,tiny_model,tiny_inputs,activate_gates,fresh_forward,
  load_k5_block,manual_single,manual_fused,CountingLinearField,tiny_bundle,
  clone_bundle,scalar_bundle,vector_from_bundle,dummy_text,show_source,
  check_sources,save_metrics,figure_path)
torch.set_num_threads(2);torch.manual_seed(42)
LAB='07_piflow_math_dx_policy';metrics={}
print('Pinned source files checked:',check_sources())


## 1. Почему clean endpoint связан с velocity

Для conditional affine path xσ=z+σ(ε−z) имеем u=ε−z, значит z=xσ−σu. После conditional expectation:

$$
\widehat z(x,\sigma)=x-\sigma u_\star(x,\sigma)
=\mathbb{E}[z\mid x_\sigma=x].
$$

Это posterior mean только для соответствующего FM parameterization и точного marginal field. Generic neural output/другая schedule не получают этот смысл автоматически.

DX student в начале сегмента (x_s,σ_s) выдаёт G параметров U_g. Source строит:

$$
a_g=x_s-\sigma_s U_g.
$$

Все U_g reparameterized относительно **одного source state**, а не являются G истинными velocities в неизвестных future states. Interpolation a(τ) задаёт dynamic clean-anchor approximation, а policy использует текущее состояние:

$$
\pi(x,\sigma)=\frac{x-a(\tau)}{\max(\sigma,\varepsilon)}.
$$

Grid anchors фиксированы внутри сегмента, но π зависит от текущего x. При новом network call строится новая policy.


In [ ]:
pm=native('core.algo.piflow_math');xs=torch.tensor([[1.2,.8]],dtype=torch.float64)
x0=torch.tensor([[.3,-.2]],dtype=torch.float64);u=xs-x0
pol=pm.DXPolicy(u[:,None].expand(1,5,2).clone(),xs,torch.ones(1,dtype=torch.float64),segment_size=1.,shift=1.)
assert torch.allclose(pol.denoising_output_x_0,x0[:,None].expand(1,5,2))
assert torch.allclose(pol.pi(xs,torch.ones(1,dtype=torch.float64)),u)
metrics['u_to_x0_max_error']=float((pol.denoising_output_x_0-x0[:,None]).abs().max())
show_source('core/algo/piflow_math.py','DXPolicy')


## 2. Raw τ и shifted σ: вывод inverse и Jacobian

$$
\sigma=f(\tau)=\frac{a\tau}{1+(a-1)\tau},\qquad a>0.
$$

Умножаем знаменатель и собираем τ: σ=τ[a−σ(a−1)]. Поэтому:

$$
\tau=f^{-1}(\sigma)=\frac{\sigma}{a+(1-a)\sigma},\qquad
f'(\tau)=\frac{a}{[1+(a-1)\tau]^2}.
$$

Если интегрировать напрямую по τ, dX/dτ=f'(τ)π(X,f(τ)). Source вместо этого выбирает raw grid, вычисляет **точное difference f(τ_next)−f(τ)** и умножает π на него. Дополнительно умножать на f' нельзя — это double counting.

`segment_size` задан по raw time. Из source τ_s получаем τ_d=max(τ_s−Δτ,0); внутренний denominator segment_size=max(τ_s−τ_d,eps). Shape broadcast добавляет singleton axes после batch.


In [ ]:
tau=torch.linspace(.01,.99,99,dtype=torch.float64);a=5.
sigma=pm.shift_timesteps(tau,a);back=sigma/(a+(1-a)*sigma)
assert torch.allclose(tau,back,atol=1e-14)
h=1e-6;fd=(pm.shift_timesteps(tau+h,a)-pm.shift_timesteps(tau-h,a))/(2*h)
jac=a/(1+(a-1)*tau).square();assert torch.allclose(fd,jac,atol=1e-8)
metrics['warp_inverse_error']=float((tau-back).abs().max());metrics['warp_derivative_error']=float((fd-jac).abs().max())
fig,axes=plt.subplots(1,2,figsize=(10,3));axes[0].plot(tau,sigma,label='sigma(tau), a=5');axes[0].plot(tau,tau,'--',label='identity');axes[0].legend();axes[0].set(xlabel='Raw tau',ylabel='Shifted sigma')
axes[1].plot(tau,jac);axes[1].set(xlabel='Raw tau',ylabel='d sigma / d tau');fig.tight_layout();fig.savefig(figure_path(LAB,'time_warp'));plt.show()


## 3. Grid interpolation, error и endpoint amplification

Grid axis идёт от destination к source: r=(τ−τ_d)/(τ_s−τ_d), q=r(G−1). После clamp floor даёт i, сосед i+1 и λ=q−i:

$$
a(\tau)=(1-\lambda)a_i+\lambda a_{i+1}.
$$

При G=1 source возвращает единственный anchor. При G≥2 endpoint r=1 выбирает последнее значение через i=G−2,λ=1. Для дважды differentiable anchor и равномерного grid spacing h_g:

$$
\|a-a_{interp}\|_\infty\leq\frac{h_g^2}{8}\sup\|a''\|.
$$

Для scalar coordinate remainder линейной интерполяции имеет вид:

$$
a(\tau)-\ell(\tau)=\tfrac12 a''(\xi)(\tau-\tau_i)(\tau-\tau_{i+1}).
$$

Произведение по модулю не больше h_g²/4; отсюда коэффициент 1/8. Для vector coordinates bound применяется покомпонентно в max norm. Velocity error из anchor error равна −δa/max(σ,eps), поэтому near-clean endpoint чувствителен к ошибкам. Уменьшение solver step не убирает policy/grid approximation error. Это мотивация dense grid/обучения, а не гарантия качества конкретного числа G.


In [ ]:
rows=[];query=torch.linspace(0,1,1001,dtype=torch.float64)
for G in [2,3,5,9,17,33]:
    times=torch.linspace(0,1,G,dtype=torch.float64);anchors=.3+.2*torch.sin(math.pi*times)
    # A query is treated as a separate batch item; all share the same coefficients.
    grids=anchors[None,:,None].expand(len(query),G,1)
    got=pm.DXPolicy._interpolate(grids,query[:,None]).flatten()
    error=float((got-(.3+.2*torch.sin(math.pi*query))).abs().max())
    bound=.2*math.pi**2/(8*(G-1)**2)
    assert error<=bound+1e-12
    rows.append({'grid':G,'max_anchor_error':error,'curvature_bound':bound})
metrics['grid_interpolation']=rows
sigma_probe=torch.tensor([1.,.1,.01,.001]);amp=.001/sigma_probe.clamp(min=1e-4)
metrics['anchor_error_amplification']={'sigma':sigma_probe.tolist(),'velocity_error_for_delta_a_0_001':amp.tolist()}
fig,ax=plt.subplots(figsize=(6,3));ax.loglog([r['grid']-1 for r in rows],[r['max_anchor_error'] for r in rows],'o-',label='Measured');ax.loglog([r['grid']-1 for r in rows],[r['curvature_bound'] for r in rows],'--',label='Curvature bound');ax.set(xlabel='G−1',ylabel='Anchor max error');ax.legend();fig.tight_layout();fig.savefig(figure_path(LAB,'grid_error'));plt.show()


### Чувствительность около clean endpoint

Ниже одна и та же anchor error 0.001 превращается в разные velocity errors. Важно показывать это отдельно от interpolation accuracy: даже маленькая grid error может быть значима при малом σ. Clamp ограничивает denominator снизу, а не исправляет predicted anchor.


In [ ]:
fig,ax=plt.subplots(figsize=(6,3));ax.loglog(sigma_probe,amp,'o-');ax.set(xlabel='Sigma',ylabel='Velocity error for anchor error 0.001',title='DX sensitivity: 1 / max(sigma, eps)');fig.tight_layout();fig.savefig(figure_path(LAB,'endpoint_sensitivity'));plt.show()


## 4. Polynomial mode: coefficients, не grid samples

Source alternative `mode='polynomial'` вычисляет:

$$
a(\tau)=\sum_{i=0}^{G-1}c_i(\tau_s-\tau)^i,\qquad
c_i=x_s-\sigma_s U_i.
$$

Здесь c_i — coefficients. Если все c_i равны z, результат обычно z(1+δτ+δτ²+...), а не constant z. Для constant polynomial нужны c_0=z,c_i=0 при i>0. Native K6 piflow sampler выбирает **grid**, polynomial — отдельная поддержанная math branch, не default released sampling recipe.

`copy` shallow: tensor storage разделяется. `detach_` отсоединяет только denoising_output_x_0, а не все tensor attributes. Это достаточно для stop-gradient по predicted policy coefficients, но x_t_src как attribute остаётся тем же объектом. Не называйте это deep clone.


In [ ]:
xs=torch.tensor([[1.2,.8]],dtype=torch.float64);x0=torch.tensor([[.3,-.2]],dtype=torch.float64)
coeff=torch.stack([x0,torch.zeros_like(x0),torch.zeros_like(x0)],1)
U=xs[:,None]-coeff
poly=pm.DXPolicy(U,xs,torch.ones(1,dtype=torch.float64),mode='polynomial')
xq=torch.tensor([[.6,.1]],dtype=torch.float64);sq=torch.tensor([.4],dtype=torch.float64)
assert torch.allclose(poly.pi(xq,sq),(xq-x0)/.4)
U=torch.randn(1,3,2,requires_grad=True)
p=pm.DXPolicy(U,xs.float(),torch.ones(1));shallow=p.copy();det=p.detach()
assert shallow.denoising_output_x_0.data_ptr()==p.denoising_output_x_0.data_ptr()
assert p.denoising_output_x_0.requires_grad and not det.denoising_output_x_0.requires_grad
assert det.x_t_src is p.x_t_src
metrics['copy_detach_contract']={'shared_storage':True,'coefficient_grad_detached':True,'deep_copy':False}


## 5. Native policy rollout: discretization vs representation

Возьмём anchor a(σ)=a₀+bσ² и dX/dσ=(X−a(σ))/σ. Разделим на σ:

$$
\frac{d}{d\sigma}\frac{X}{\sigma}=-\frac{a_0}{\sigma^2}-b.
$$

Интеграция от 1 до σ даёт exact trajectory:

$$
X(\sigma)=a_0+\sigma(X_1-a_0+b)-b\sigma^2.
$$

Сравним native Euler-policy rollout до σ=0.2, а не только до 0: endpoint DX может скрывать intermediate trajectory error. При fixed grid всё более мелкий solver сначала уменьшает discretization error, затем упирается в representation error.

Source выбирает n_b=max(1,round((τ_s−τ_d)·total_substeps)); разные элементы batch могут иметь разные n. Loop идёт до max(n_b), `torch.where(active_mask,...)` замораживает завершённые элементы.


In [ ]:
a0=.3;b=.2;xs=torch.tensor([[1.2]],dtype=torch.float64);rows=[]
for G in [3,9,33]:
    grid_t=torch.linspace(0,1,G,dtype=torch.float64)
    anchors=(a0+b*grid_t.square())[None,:,None]
    p=pm.DXPolicy(xs[:,None]-anchors,xs,torch.ones(1,dtype=torch.float64),segment_size=1.)
    for steps in [8,32,128,512]:
        out,sig,_=pm.policy_rollout_fm(xs,torch.ones(1,dtype=torch.float64),torch.ones(1,dtype=torch.float64),torch.tensor([.2],dtype=torch.float64),steps,p)
        exact=a0+.2*(1.2-a0+b)-b*.2**2
        rows.append({'grid':G,'density':steps,'error':abs(float(out)-exact)})
metrics['policy_rollout_error']=rows
assert next(r['error'] for r in rows if r['grid']==33 and r['density']==512)<next(r['error'] for r in rows if r['grid']==33 and r['density']==8)
fig,ax=plt.subplots(figsize=(6,3.5))
for G in [3,9,33]:
    rr=[r for r in rows if r['grid']==G];ax.loglog([r['density'] for r in rr],[r['error'] for r in rr],'o-',label=f'G={G}')
ax.set(xlabel='Policy substep density over unit raw time',ylabel='Error at sigma=0.2');ax.legend();fig.tight_layout();fig.savefig(figure_path(LAB,'rollout_error'));plt.show()


## 6. Разная длина rollout в одном batch

Source выполняет max(n_b) vectorized policy calls, а updates для завершённых rows отключает mask. Это отличается от суммы scalar calls по всем batch rows. Для одного grid anchor G=1 interpolation возвращает его без gather. Проверим обе ветви и terminal σ каждого элемента.


In [ ]:
single=torch.tensor([[[.3,.4]]]);assert torch.equal(pm.DXPolicy._interpolate(single,torch.tensor([[.8]])),single[:,0])
ss=torch.tensor([1.,.5],dtype=torch.float64);end=torch.tensor([.4,.25],dtype=torch.float64)
x0=torch.tensor([[.3],[.1]],dtype=torch.float64);u=torch.tensor([[.8],[-.3]],dtype=torch.float64)
xs=x0+ss[:,None]*u
p=pm.DXPolicy(u[:,None].expand(2,3,1).clone(),xs,ss,segment_size=ss-end)
class CountPolicy:
    shift=1.
    def __init__(self,policy):self.policy=policy;self.calls=0
    def pi(self,x,sigma):self.calls+=1;return self.policy.pi(x,sigma)
proxy=CountPolicy(p)
out,sigma,last_time=pm.policy_rollout_fm(xs,ss,ss,end,20,proxy)
assert torch.allclose(out,x0+end[:,None]*u,atol=1e-12)
assert torch.allclose(sigma.flatten(),end) and proxy.calls==12
metrics['heterogeneous_rollout']={'vectorized_policy_calls':proxy.calls,'active_steps_per_row':[12,5],
 'endpoint_error':float((out-x0-end[:,None]*u).abs().max()),'network_time_at_end':last_time.tolist()}


## Измерение → вывод

Все методы DXPolicy и shift/rollout проверены по формулам. Grid error, solver error и near-endpoint amplification различаются. Полный π-Flow paper включает и GMFlow policy; данный K6 source реализует DX, поэтому GMFlow theorem на него автоматически не переносится.


## Самостоятельные опыты

1. Выведите inverse warp и dx/dτ заново; где должен стоять Jacobian?
2. Проверьте G=1, r<0, r>1 и endpoint interpolation.
3. Сравните grid и polynomial для constant anchor.
4. Добавьте batch с разными durations rollout и проверьте active_mask.
5. Почему policy approximation error усиливается около σ=0?

Перед изменением запишите прогноз. Сохраняйте baseline, меняйте один фактор, отмечайте seed и tensor layout. Нулевая ошибка на oracle не равна pretrained quality.


In [ ]:
result=save_metrics(LAB,metrics)
print(json.dumps(result,ensure_ascii=False,indent=2))


## Первичные источники

[MIT course / notes](https://diffusion.csail.mit.edu/2026/index.html) · [DiT](https://arxiv.org/abs/2212.09748) · [Flow Matching](https://arxiv.org/abs/2210.02747) · [K5](https://arxiv.org/abs/2511.14993) · [K6 §3.2, §4.5](https://arxiv.org/html/2610.05608v1) · [π-Flow §3.1, §4 / Alg.1](https://arxiv.org/html/2510.14974v1) · [Source hashes / provenance](../sources.json).

Формулы ниже/выше выводятся для явно заданных соглашений. Тела source methods сохраняются; adapters и doubles подписаны. Малые experiments не воспроизводят промышленное обучение.
